In [48]:
# https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client.models import Distance, VectorParams

In [49]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from pathlib import Path


In [50]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [51]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
embeddings = HuggingFaceEndpointEmbeddings(client="http://localhost:8080")

In [52]:
client = QdrantClient(url="http://localhost:6333")

In [53]:
md_dir = (Path().resolve() / "md_pages_clean").resolve()

In [54]:
md_dir

WindowsPath('C:/Users/jw160/project/RAG/notebooks/md_pages_clean')

In [55]:
loader = DirectoryLoader(
    str(md_dir),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

In [56]:
docs = loader.load()

In [ ]:
# 범용 질문 품질을 위해 분할 기준을 문단 중심으로 바꾸고,
# 임베딩 모델 입력 한도에 맞추기 위해 토큰 길이를 사용합니다.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def token_len(text: str) -> int:
    # special token 포함 길이를 사용해 임베딩에서의 truncate 리스크를 줄입니다.
    return len(embeddings.encode(text, add_special_tokens=True))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=110,
    chunk_overlap=20,
    length_function=token_len,
    is_separator_regex=False,
    separators=["\n\n", "\n", " "],
)

In [58]:
texts = text_splitter.create_documents(
    [doc.page_content for doc in docs],
    metadatas=[doc.metadata for doc in docs]
)

AttributeError: 'HuggingFaceEndpointEmbeddings' object has no attribute 'encode'

In [21]:
texts[:3]

[Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai.md'}, page_content='# https://www.heum.ai\n\n[![logo](https://opening-attachments.greetinghr.com/2025-07-15/33a08bfb-0ad7-4c6b-89ce-9ef0800014c5/Alfred-Logo-Hori-Color.png)](https://www.heum.ai/ko)\n[Company](https://www.heum.ai/ko/home)\n[Heum way](https://www.heum.ai/ko/culture)\n[FAQ](https://www.heum.ai/ko/apply-faq)\n[Recruit](https://www.heum.ai/ko/apply)\n[Blog](https://www.alfred.kr/blog)\n|\n[Alfred ↗️](https://www.alfred.kr)\n사업자를 위한   \n금융 에이전트 기업 \nHeum\nHeum은 초격차 기술을 기반으로 \n사업자를 위한 경리/급여/세무 에이전트를 개발합니다.\n슈퍼 에이전트가\n만들어 낼 새로운 세상'),
 Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai.md'}, page_content='슈퍼 에이전트가\n만들어 낼 새로운 세상\nHeum은 누구나 나만의 금융 비서, \n슈퍼 에이전트를 가지는 세상을 꿈꿉니다.\n사업자를 위한 \n나만의 경리/급여/세무 비서, 알프레드\n배트맨의 집사 알프레드처럼 에이전트 알프레드가\n세무에서 재무까지 사업자의 금융을 돕습니다\n더 알아보기\n알프레드는 슈퍼 앱을 넘어\n슈퍼 에이전트에\n도전합니다\n배트맨의 집사 알프레드처럼 사업자를 알아서 챙깁

In [23]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

In [25]:
# 범용 질의 응답 품질을 위해 헤더 기반 2차 분할을 생략하고,
# 1차 청킹 결과(texts)를 그대로 사용합니다.
md_header_splits = texts

In [27]:
md_header_splits[:3]

[Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai.md', 'Header 1': 'https://www.heum.ai'}, page_content='[![logo](https://opening-attachments.greetinghr.com/2025-07-15/33a08bfb-0ad7-4c6b-89ce-9ef0800014c5/Alfred-Logo-Hori-Color.png)](https://www.heum.ai/ko)\n[Company](https://www.heum.ai/ko/home)\n[Heum way](https://www.heum.ai/ko/culture)\n[FAQ](https://www.heum.ai/ko/apply-faq)\n[Recruit](https://www.heum.ai/ko/apply)\n[Blog](https://www.alfred.kr/blog)\n|\n[Alfred ↗️](https://www.alfred.kr)\n사업자를 위한\n금융 에이전트 기업\nHeum\nHeum은 초격차 기술을 기반으로\n사업자를 위한 경리/급여/세무 에이전트를 개발합니다.\n슈퍼 에이전트가\n만들어 낼 새로운 세상'),
 Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai.md'}, page_content='슈퍼 에이전트가\n만들어 낼 새로운 세상\nHeum은 누구나 나만의 금융 비서,\n슈퍼 에이전트를 가지는 세상을 꿈꿉니다.\n사업자를 위한\n나만의 경리/급여/세무 비서, 알프레드\n배트맨의 집사 알프레드처럼 에이전트 알프레드가\n세무에서 재무까지 사업자의 금융을 돕습니다\n더 알아보기\n알프레드는 슈퍼 앱을 넘어\n슈퍼 에이전트에\n도전합니다\n배트맨의 집사 알프레드처럼 사업자를 알아서

In [34]:
# 모든 문서의 임베딩 벡터 사이즈가 다를 수 있으므로,
# 가장 많이 등장하는 사이즈(또는 전체 문서의 벡터 사이즈가 동일한지 체크)로 컬렉션 생성.
# from collections import Counter

# vector_sizes = [len(embeddings.embed_query(doc.page_content)) for doc in md_header_splits]
# size_counts = Counter(vector_sizes)

# most_common_size, count = size_counts.most_common(1)[0]
# if len(size_counts) > 1:
#     print(f"Warning: Detected multiple embedding sizes: {size_counts}. Using most common size: {most_common_size}")

# if not client.collection_exists("test2"):
#     client.create_collection(
#         collection_name="test2",
#         vectors_config=VectorParams(size=most_common_size, distance=Distance.COSINE)
#     )

In [35]:
vector_size = len(embeddings.embed_query(md_header_splits[0].page_content))


In [36]:
# NOTE: 기존 `test3`(dense-only) 컬렉션 생성 로직은
# hybrid 저장(from_documents) 단계에서 새 컬렉션(`test3_hybrid`)을 만들기 때문에 비활성화합니다.
pass

In [37]:
# HYBRID 컬렉션(새 컬렉션) 생성 + dense+sparse 인덱싱
hybrid_collection_name = "test3_hybrid"

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

vector_store = QdrantVectorStore.from_documents(
    md_header_splits,
    embedding=embeddings,
    sparse_embedding=sparse_embeddings,
    location=client,
    collection_name=hybrid_collection_name,
    retrieval_mode=RetrievalMode.HYBRID,  # 기본 Fusion = RRF
)

In [42]:
# from_documents()가 이미 md_header_splits 전체를 dense+sparse로 인덱싱합니다.
pass

In [43]:
query = "채용공고"
result = vector_store.similarity_search(query, k=4)

In [46]:
result

In [47]:
result

[Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai_ko_o_121515.md', '_id': '60fbf032-8442-42c0-8005-7e0229d9571f', '_collection_name': 'test3'}, page_content='* 채널톡 및 고객 상담 지원'),
 Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai_o_121515.md', '_id': 'd4c2638b-7f3f-48ae-971e-7715b9ae1020', '_collection_name': 'test3'}, page_content='* 채널톡 및 고객 상담 지원'),
 Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai_o_115332.md', 'Header 3': '**PO/PM(주니어/미들/시니어/리더/트라이브리더/CPO)**', '_id': '6ad445e8-5cb5-4354-ac11-1c53275809ac', '_collection_name': 'test3'}, page_content='* 서비스 기획 혹은 리딩'),
 Document(metadata={'source': 'C:\\Users\\jw160\\project\\RAG\\notebooks\\md_pages\\https___www_heum_ai_ko_o_115332.md', 'Header 3': '**PO/PM(주니어/미들/시니어/리더/트라이브리더/CPO)**', '_id': 'c9e83b55-9894-4479-9f32-ef1262cec618', '_collection_name': 'test3'}, page_content